# 18 - Pass 9: do the three working ingredients stack? (one commit, GPU)

Same structure as notebooks 08 and 11-17: fetch the code from GitHub, run the tests, check the data on
the CPU, train two runs at a time (one per GPU), then **the analysis runs inside this notebook** so the
result can be judged straight away.

**What Pass 9 asks** (vault: Experiment Plan, "Pass 9 plan"): at 1% labels, three ingredients each gave
about +1.6 macro-F1 over plain BERT, and they were never combined:

- **helper heads on L** (A2): sentiment and emotion heads trained on the labeled headlines only;
- **FGM** (A7): a small adversarial push on the word embeddings at every step;
- **self-training** (A6): a first model labels the unlabeled headlines it is sure about, and a fresh
  model trains on the labeled plus those pseudo-labeled headlines.

Do they add up, or do they cover the same ground? Pass 9 fills the four missing cells of a 2x2x2 design:

| Arm | Helpers on L | FGM | Self-training | Exists already? |
|---|---|---|---|---|
| A1 | | | | Pass 1 |
| A2 | yes | | | Pass 2a |
| A7 | | yes | | Pass 2b (100%: Pass 7) |
| A6 | | | yes | Pass 2a |
| **A27** | yes | yes | | **new** |
| **A28** | yes | | yes | **new** |
| **A29** | | yes | yes | **new** |
| **A30** | yes | yes | yes | **new** |

Settings are the frozen ones of the single arms (helper weight 0.3, FGM epsilon 0.5, pseudo-label
threshold 0.9). In the self-training arms, the first model (the "teacher") uses the arm's own recipe
without self-training. **Nothing is picked on test.** 5 seeds, 1% and 10% labels; with `RUN_FULL100`,
also A27 with every label (self-training needs unlabeled headlines, so it has no 100% cell).

**Before you run, in the right-hand panel:**
- **Input:** the same `gainsplit-processed` version as notebooks 10-17.
- **Internet: on**, **Accelerator: GPU T4 x2**.
- Then **Save Version -> Save & Run All (Commit)**. About 1.5 hours without the 100% runs, about 2.5
  hours with them; you can close the browser.

## 1. Get the code
Clones the public repo and prints the **commit**. Every run records it in its `result.json`.

In [ ]:
import os, subprocess, sys, time
T_START = time.time()
REPO_URL = "https://github.com/Ariffurrhmn/GainSplit.git"
REPO_DIR = "/tmp/GainSplit"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
sys.path.insert(0, f"{REPO_DIR}/src")
COMMIT = subprocess.check_output(["git", "-C", REPO_DIR, "rev-parse", "HEAD"], text=True).strip()
print("code commit:", COMMIT)

## 2. Tests (about two minutes)
The repo's tests train a tiny random BERT for a few steps on made-up data. For Pass 9 they also train
the four stacked arms, check that the self-training teacher uses the right recipe, and check that the
helper heads never see the pseudo-labeled rows. They prove the pieces fit together before real GPU
time is spent. If anything fails, stop and paste the output.

In [ ]:
r = subprocess.run([sys.executable, "-m", "pytest", "tests", "-q", "-p", "no:cacheprovider"],
                   cwd=REPO_DIR, capture_output=True, text=True)
print(r.stdout[-4000:], r.stderr[-2000:])
assert r.returncode == 0, "tests failed: see output above"

## 3. Check the data on the CPU (pre-flight)
Checks every file against `manifest.json`. Any problem raises an error here, before GPU time is spent.
`RUN_FULL100 = True` adds the 5 runs of A27 with every label (about 1 hour of the notebook's time); set it
to False only if quota is short.

In [ ]:
import torch
from gainsplit.data import find_data_dir, verify, preflight
from gainsplit.plans import PLANS

PLAN = "pass9"
RUN_FULL100 = True
OUT_DIR = f"/kaggle/working/results/{PLAN}"
configs = (PLANS["pass9_100"]() if RUN_FULL100 else []) + PLANS[PLAN]()   # longest first

print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
DATA_DIR = find_data_dir("/kaggle/input")
verify(DATA_DIR)
print("data:", DATA_DIR, "- all files match manifest.json")
display(preflight(configs, DATA_DIR))

## 4. The list of runs and the time estimate
48 runs with `RUN_FULL100` (43 without):
- A27 at 100% x5 first (about 25 min each);
- then the four new arms at 10% (about 4-9 min each: the self-training arms train two models, and FGM
  doubles the work per step) and at 1% (about 1-3 min each);
- last, one re-run each of A2, A6 and A7 at 1%, seed 0, tag `rerun`. The training code of all three
  changed, so each must reproduce its Pass 2 run exactly.

In [ ]:
for c in configs:
    print(f"{c.run_id:18s} arm={c.arm:3s} labels={c.budget:3d}% seed={c.seed} helper_w={c.helper_weight:g} eps={c.fgm_epsilon:g}")
print(len(configs), "runs")

## 5. Run
One run per GPU; when a GPU frees up, the next run starts. Each run's log is in `results/<plan>/logs/<run_id>.log`.
If the session stops halfway, run this cell again: finished runs are skipped.

In [ ]:
from gainsplit.runner import run_all
table = run_all(configs, DATA_DIR, OUT_DIR)
table.to_csv(f"{OUT_DIR}/summary.csv", index=False)
print(f"total so far: {(time.time() - T_START) / 60:.1f} min")

## 6. Checks and analysis (CPU, about 3 minutes)
1. **Re-run check:** A2, A6 and A7 (1%, seed 0) must equal their Pass 2 runs exactly.
2. **Means per cell** (points) for all eight cells of the design.
3. **Additivity** (a reading aid, no test): each combination's gain over BERT next to the sum of its
   single-ingredient gains. `share_of_sum` near 1 = they add up; near 0 = the extra ingredients add nothing.
4. **Paired bootstrap** over test headlines (10,000 resamples, mean over the 5 seeds), Holm-corrected within
   each table: `contrasts_f1` (what each addition gains) and `interactions` (whether gains add up: 0 = they
   add exactly, negative = they overlap, positive = they reinforce each other). Contrasts are fixed in
   `src/gainsplit/analysis_pass9.py`, written before any run.
5. **Self-training health:** the teacher's dev score and how many pseudo-labels it kept, per arm.

In [ ]:
from pathlib import Path
from gainsplit import analysis_pass9 as a9
import pandas as pd
pd.set_option("display.width", 200)
RESULTS_DIRS = [d for d in [f"{REPO_DIR}/results/{p}" for p in ("pass1", "pass2a", "pass2b", "pass7")] + [OUT_DIR]
                if Path(d).exists()]
print("results folders:", RESULTS_DIRS)
ANALYSIS_DIR = f"{OUT_DIR}/analysis"
tables = a9.run_all(RESULTS_DIRS, ANALYSIS_DIR)
for name, t in tables.items():
    if name != "verdicts":
        print(f"\n=== {name}")
        display(t)
rr = tables.get("rerun_check")
rr9 = None if rr is None else rr[rr["arm"].isin(["A2", "A6", "A7"]) & (rr["budget"] == 1)]
print("re-run check:", "PASS" if rr9 is not None and len(rr9) == 3 and rr["ok"].all() else "DIFFERENCE or missing - report this first")

## 7. Verdict against the predictions written before running
Each prediction counts as **supported** only if the result goes the predicted way **and** passes Holm
(p_holm < 0.05); "no difference" predictions are supported when p_holm >= 0.05. The predictions and
their reasoning are in the vault (Experiment Plan, "Pass 9 plan"). Paste this cell's output when you report.

In [ ]:
v = tables["verdicts"]
for _, r in v.iterrows():
    if pd.isna(r.get("mean_diff")):
        print(f"{r.hypothesis} {r.contrast} at {r.budget}%: {r.verdict}")
        continue
    print(f"{r.hypothesis} {r.contrast:38s} {r.budget:3d}%  {r.mean_diff:+.2f} points [{r.ci_low:+.2f}, {r.ci_high:+.2f}]  "
          f"p_holm {r.p_holm:.3f}  -> {r.verdict}   ({r.claim})")
if len(tables["contrasts_accuracy_100"]):
    print("\naccuracy at 100% (the number the seed paper reports):")
    display(tables["contrasts_accuracy_100"])

## Download
Zips the results folder (runs, logs, pretraining receipt, analysis tables). A commit keeps it in the
Output tab: download `results_<plan>.zip` and send it with the notebook log.

In [ ]:
import shutil
print(shutil.make_archive(f"/kaggle/working/results_{PLAN}", "zip", OUT_DIR))
print(f"notebook time: {(time.time() - T_START) / 60:.1f} min")